# Sri Lanka Live Broiler Price — Time Series Forecasting

**Data:** daily live broiler price (Rs/kg), the *mode* across companies on each "Live Price" board posted by
Alpha Broiler Farm Association of Sri Lanka, 12 Dec 2024 → 29 Oct 2026.

**Workflow**
1. Setup & config
2. Data loading & quality checks
2b. **Preprocessing** — outliers, imputation (method chosen by experiment), transformation, calendar features
3. Exploratory analysis (trend, weekday effect, monthly pattern, volatility)
4. Stationarity tests (ADF, KPSS) & decomposition (STL)
5. ACF / PACF
6. Train / test split (chronological hold-out)
7. Baseline models (naive, seasonal naive, drift, moving average)
8. ARIMA — order chosen by AIC grid search
9. SARIMA — weekly seasonality (s = 7)
10. SARIMAX — ARIMA + holiday / day-of-week regressors
11. Exponential smoothing (ETS / Holt)
12. ML benchmark — gradient boosting on lag features
13. Hold-out evaluation (MAE, RMSE, MAPE, sMAPE, MASE) — **scored only on real posted days**
14. Rolling-origin (walk-forward) cross-validation at 1, 7 and 14-day horizons
15. Residual diagnostics of the best statistical model
16. Final fit on all data + 30-day forecast with prediction intervals
17. Summary tables to share

> Key design choice: ~40% of calendar days had no post and were interpolated. Models are **trained** on the
> continuous interpolated series (ARIMA needs a regular index), but **errors are measured only on days with a real
> board**, so interpolation can't flatter the scores.

## 1. Setup & configuration

In [ ]:
import warnings, itertools, time
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.ensemble import GradientBoostingRegressor

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.figsize": (13, 4.5), "axes.grid": True, "grid.alpha": .3})

# ---------------- CONFIG ----------------
DATA_PATH   = "live_broiler_price_daily_2024-12_to_2026-10.csv"   # change if the file is elsewhere
TARGET      = "price_interpolated"     # continuous series used for fitting
OBSERVED    = "mode_price_lkr_per_kg"  # real posted values used for scoring
TEST_DAYS   = 60                       # final hold-out length (calendar days)
SEASON      = 7                        # weekly seasonality
FORECAST_DAYS = 30
RANDOM_STATE  = 42
np.random.seed(RANDOM_STATE)

## 2. Data loading & quality checks

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["date"]).set_index("date").sort_index()
df = df.asfreq("D")   # enforce a regular daily index

print("Shape:", df.shape)
print("Range:", df.index.min().date(), "->", df.index.max().date(), f"({len(df)} days)")
print("\nStatus counts:\n", df["status"].value_counts())
print("\nMissing per column:\n", df.isna().sum())
print("\nDuplicate dates:", df.index.duplicated().sum())
df.head(10)

In [ ]:
obs = df[OBSERVED].dropna()
print(obs.describe().round(1))

# Longest run of consecutive days without a real post
gap = df[OBSERVED].isna().astype(int)
runs = gap.groupby((gap != gap.shift()).cumsum()).sum()
print("\nLongest gap without a board:", runs.max(), "days")
print("Share of days observed: {:.1%}".format(df[OBSERVED].notna().mean()))

## 2b. Preprocessing
| Step | What | Why |
|---|---|---|
| P1 | Regular daily index, dtype checks | ARIMA/ETS need equally spaced data |
| P2 | Holiday & no-post days → missing (never 0) | A closed market is not a zero price |
| P3 | Outlier detection (Hampel filter) | Separate real market shocks from reading errors |
| P4 | Missing-value imputation — methods compared by a masking experiment | Choose the filler with evidence, not by habit |
| P5 | Variance check & transformation (log / Box-Cox) | Stabilise variance if errors grow with price level |
| P6 | Calendar features (holiday, after-holiday, weekday) | Exogenous inputs for SARIMAX / ML |
| P7 | Final model-ready series | One clean series used by every model below |

In [ ]:
# ---------- P1. regular index & types ----------
raw = df[OBSERVED].astype(float)                 # ONLY real board values, NaN elsewhere
assert raw.index.is_monotonic_increasing and raw.index.freq == "D"
assert (raw.dropna() > 0).all(), "non-positive prices found"
print("P1 ok — daily index, positive prices. Real observations:", raw.notna().sum())

# ---------- P2. holiday / no-post days stay missing ----------
is_obs     = raw.notna()
is_holiday = (df["status"] == "holiday").astype(int)
print("P2 — holidays:", is_holiday.sum(), "| no-post days:", (df['status'] == 'no_post').sum(), "| both kept as NaN, not 0")

In [ ]:
# ---------- P3. outlier detection: Hampel filter on real observations ----------
OUTLIER_TREATMENT = "keep"   # "keep" (default: broiler price jumps are genuine supply shocks) or "clip"
HAMPEL_WINDOW, HAMPEL_K = 7, 3.0

o = raw.dropna()
med = o.rolling(HAMPEL_WINDOW, center=True, min_periods=3).median()
mad = (o - med).abs().rolling(HAMPEL_WINDOW, center=True, min_periods=3).median() * 1.4826
outlier = (o - med).abs() > HAMPEL_K * mad.replace(0, np.nan)
print(f"P3 — Hampel flagged {outlier.sum()} of {len(o)} observations")
display(pd.DataFrame({"price": o, "rolling_median": med.round(1), "deviation": (o - med).round(1)})[outlier])

fig, ax = plt.subplots()
ax.plot(o, lw=1, color="tab:blue", label="posted"); ax.plot(med, lw=1, color="orange", label="rolling median")
ax.scatter(o.index[outlier], o[outlier], color="red", s=40, zorder=5, label="flagged")
ax.set_title("Outlier check (Hampel filter)"); ax.legend(); plt.show()

clean = raw.copy()
if OUTLIER_TREATMENT == "clip":
    lo, hi = med - HAMPEL_K * mad, med + HAMPEL_K * mad
    clean.loc[o.index] = o.clip(lo, hi)
    print("Outliers clipped to the Hampel band")
else:
    print("Outliers kept — review the table above. Sudden jumps in broiler prices are usually real supply shocks;\n"
          "set OUTLIER_TREATMENT = 'clip' only if a flagged value looks like a reading error.")

In [ ]:
# ---------- P4. imputation: compare methods by hiding known values ----------
def impute(s, method):
    if method == "linear":   return s.interpolate("linear")
    if method == "time":     return s.interpolate("time")
    if method == "ffill":    return s.ffill()
    if method == "spline":   return s.interpolate("spline", order=3)
    if method == "pchip":    return s.interpolate("pchip")
    if method == "akima":    return s.interpolate("akima")
    raise ValueError(method)

METHODS = ["linear", "time", "ffill", "spline", "pchip", "akima"]
rng = np.random.default_rng(RANDOM_STATE)
inner = clean.dropna().index[5:-5]
scores = {m: [] for m in METHODS}
for rep in range(20):                                   # 20 random masks of 15% of real points
    hide = rng.choice(inner, size=int(.15 * len(inner)), replace=False)
    masked = clean.copy(); masked.loc[hide] = np.nan
    for m in METHODS:
        try:
            filled = impute(masked, m)
            scores[m].append(np.mean(np.abs(filled.loc[hide] - clean.loc[hide])))
        except Exception:
            pass
imp_tbl = pd.DataFrame({m: [np.mean(v), np.std(v)] for m, v in scores.items() if v}, index=["MAE", "std"]).T.sort_values("MAE").round(2)
BEST_IMPUTE = imp_tbl.index[0]
print("P4 — imputation error on hidden real values (Rs/kg):"); display(imp_tbl)
print("Selected imputation:", BEST_IMPUTE)

y = impute(clean, BEST_IMPUTE).ffill().bfill()
y.name = "price"
print("NaN after imputation:", y.isna().sum())

In [ ]:
# ---------- P5. variance stabilisation ----------
from scipy import stats
TRANSFORM = "auto"   # "auto", "log" or "none"

lv = pd.DataFrame({"level_30d_mean": y.rolling(30).mean(),
                   "volatility_30d_std": y.diff().rolling(30).std()}).dropna()
mean_sd_corr = lv.corr().iloc[0, 1]
bc_lambda = stats.boxcox_normmax(y.values, method="mle")
print(f"P5 — corr(rolling level, rolling volatility) = {mean_sd_corr:.2f}")
print(f"     Box-Cox lambda (MLE) = {bc_lambda:.2f}   (1 = no transform, 0 = log)")

if TRANSFORM == "auto":
    TRANSFORM = "log" if mean_sd_corr > 0.3 else "none"
print("     Transformation used:", TRANSFORM)

if TRANSFORM == "log":
    T, IT = np.log, np.exp
else:
    T = IT = (lambda v: v)

fig, axes = plt.subplots(1, 2, figsize=(15, 3.5))
lv.plot.scatter(x="level_30d_mean", y="volatility_30d_std", ax=axes[0], title="Level vs volatility (30-day windows)")
T(y).plot(ax=axes[1], title=f"Model-scale series ({TRANSFORM})"); plt.show()

In [ ]:
# ---------- P6. calendar features ----------
def make_exog(index):
    X = pd.DataFrame(index=index)
    hol = is_holiday.reindex(index).fillna(0)
    X["holiday"] = hol
    X["after_holiday"] = hol.shift(1).fillna(0)
    dows = pd.get_dummies(index.dayofweek, prefix="dow", drop_first=True).astype(float)
    dows.index = index
    return pd.concat([X, dows], axis=1).astype(float)
X_all = make_exog(y.index)
print("P6 — exogenous features:", list(X_all.columns))

# ---------- P7. final model-ready data ----------
prep_summary = pd.Series({
    "days": len(y), "real observations": int(is_obs.sum()), "imputed days": int((~is_obs).sum()),
    "imputation": BEST_IMPUTE, "outliers flagged": int(outlier.sum()), "outlier treatment": OUTLIER_TREATMENT,
    "transform": TRANSFORM, "exog features": X_all.shape[1]})
print("\nP7 — preprocessing summary"); print(prep_summary.to_string())

## 3. Exploratory data analysis

In [ ]:
fig, ax = plt.subplots()
ax.plot(y.index, y, color="lightgray", lw=1.2, label="interpolated (training series)")
ax.scatter(obs.index, obs, s=9, color="tab:blue", label="posted board value", zorder=3)
hol = df.index[df["status"] == "holiday"]
ax.scatter(hol, y.loc[hol], marker="x", color="red", s=40, label="holiday (no price)", zorder=4)
ax.plot(y.rolling(30, center=True).mean(), color="tab:orange", lw=2, label="30-day rolling mean")
ax.set_title("Live broiler price (Rs/kg)"); ax.set_ylabel("Rs/kg"); ax.legend(loc="upper left")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
obs.plot(kind="hist", bins=25, ax=axes[0], title="Distribution of posted prices")
dow = obs.groupby(obs.index.dayofweek).agg(["mean", "count"])
dow.index = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
dow["mean"].plot(kind="bar", ax=axes[1], title="Mean price by weekday (posted days)")
axes[1].set_ylim(obs.min() * .95, obs.max())
m = obs.to_frame("p"); m["month"] = m.index.to_period("M").astype(str)
m.boxplot(column="p", by="month", ax=axes[2], rot=90); axes[2].set_title("Monthly distribution"); plt.suptitle("")
plt.tight_layout(); plt.show()
print(dow)

In [ ]:
# Day-to-day volatility on consecutive observed days
d = obs.diff()[obs.index.to_series().diff().dt.days == 1]
print("Daily change on consecutive posted days:")
print(d.describe().round(1))
print("Days with |change| >= 50 Rs:", (d.abs() >= 50).sum())
y.diff().rolling(30).std().plot(title="30-day rolling std of daily change (volatility)"); plt.show()

## 4. Stationarity tests & decomposition

In [ ]:
def stationarity_report(series, name):
    s = series.dropna()
    adf_p = adfuller(s, autolag="AIC")[1]
    kpss_p = kpss(s, regression="c", nlags="auto")[1]
    verdict = "stationary" if (adf_p < .05 and kpss_p > .05) else "NON-stationary" if (adf_p > .05 and kpss_p < .05) else "inconclusive"
    return {"series": name, "ADF p (H0: unit root)": round(adf_p, 4), "KPSS p (H0: stationary)": round(kpss_p, 4), "verdict": verdict}

st = pd.DataFrame([
    stationarity_report(T(y), "level"),
    stationarity_report(T(y).diff(), "1st difference"),
    stationarity_report(T(y).diff().diff(SEASON), "1st diff + weekly diff"),
])
st

In [ ]:
stl = STL(y, period=SEASON, robust=True).fit()
fig = stl.plot(); fig.set_size_inches(13, 8); plt.show()
var_resid = np.var(stl.resid)
print("Strength of trend    : {:.3f}".format(max(0, 1 - var_resid / np.var(stl.trend + stl.resid))))
print("Strength of weekly s.: {:.3f}  (close to 0 = weak weekly seasonality)".format(max(0, 1 - var_resid / np.var(stl.seasonal + stl.resid))))

## 5. ACF / PACF

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 7))
plot_acf(y, lags=40, ax=axes[0, 0], title="ACF — level")
plot_pacf(y, lags=40, ax=axes[0, 1], title="PACF — level", method="ywm")
plot_acf(y.diff().dropna(), lags=40, ax=axes[1, 0], title="ACF — 1st difference")
plot_pacf(y.diff().dropna(), lags=40, ax=axes[1, 1], title="PACF — 1st difference", method="ywm")
plt.tight_layout(); plt.show()

## 6. Train / test split (chronological)

In [ ]:
train, test = y.iloc[:-TEST_DAYS], y.iloc[-TEST_DAYS:]
test_obs_mask = is_obs.loc[test.index]            # score only real posted days
print(f"Train: {train.index.min().date()} -> {train.index.max().date()}  ({len(train)} days)")
print(f"Test : {test.index.min().date()} -> {test.index.max().date()}  ({len(test)} days, {test_obs_mask.sum()} with a real board)")

ax = train.plot(label="train"); test.plot(ax=ax, label="test"); ax.legend(); ax.set_title("Hold-out split"); plt.show()

In [ ]:
# ---------- metrics ----------
# MASE scale = mean |change| between consecutive REAL posted values in train
# (using the interpolated series would understate volatility and inflate MASE)
def mase_scale(observed_train):
    return np.mean(np.abs(np.diff(observed_train.values)))

SCALE = mase_scale(df.loc[train.index, OBSERVED].dropna())
print(f"MASE scale (naive error between consecutive posts): {SCALE:.2f} Rs/kg")

def metrics(actual, pred, scale=SCALE):
    a, p = np.asarray(actual, float), np.asarray(pred, float)
    e = a - p
    return {
        "MAE":   np.mean(np.abs(e)),
        "RMSE":  np.sqrt(np.mean(e ** 2)),
        "MAPE%": np.mean(np.abs(e / a)) * 100,
        "sMAPE%":np.mean(2 * np.abs(e) / (np.abs(a) + np.abs(p))) * 100,
        "MASE":  np.mean(np.abs(e)) / scale,
        "Bias":  np.mean(p - a),
    }

def score(pred, name):
    pred = pd.Series(np.asarray(pred, float), index=test.index)
    actual = df.loc[test.index, OBSERVED][test_obs_mask]
    r = metrics(actual, pred[test_obs_mask]); r["model"] = name
    return r

FORECASTS, RESULTS = {}, []

## 7. Baseline models
Any real model must beat these to be worth using.

In [ ]:
h = len(test)
base = {
    "Naive (last value)":   np.repeat(train.iloc[-1], h),
    "Seasonal naive (7d)":  np.resize(train.iloc[-SEASON:].values, h),
    "Moving average (7d)":  np.repeat(train.iloc[-7:].mean(), h),
    "Drift":                train.iloc[-1] + (train.iloc[-1] - train.iloc[0]) / (len(train) - 1) * np.arange(1, h + 1),
}
for k, v in base.items():
    FORECASTS[k] = pd.Series(v, index=test.index); RESULTS.append(score(v, k))
pd.DataFrame(RESULTS).set_index("model").round(2)

## 8. ARIMA — AIC grid search

In [ ]:
def grid_search(series, orders, seasonal_orders=[(0,0,0,0)], exog=None, top=5):
    rows = []
    for order in orders:
        for sorder in seasonal_orders:
            try:
                t0 = time.time()
                res = SARIMAX(series, exog=exog, order=order, seasonal_order=sorder,
                              enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
                rows.append({"order": order, "seasonal": sorder, "AIC": res.aic, "BIC": res.bic, "sec": round(time.time() - t0, 2)})
            except Exception as e:
                pass
    return pd.DataFrame(rows).sort_values("AIC").head(top).reset_index(drop=True)

arima_orders = [(p, 1, q) for p in range(0, 4) for q in range(0, 4)]
arima_grid = grid_search(T(train), arima_orders)
arima_grid

In [ ]:
best_arima_order = arima_grid.loc[0, "order"]
arima_res = SARIMAX(T(train), order=best_arima_order).fit(disp=False)
print(arima_res.summary().tables[1])
fc = arima_res.get_forecast(h)
FORECASTS[f"ARIMA{best_arima_order}"] = IT(fc.predicted_mean)
ARIMA_CI = IT(fc.conf_int())
RESULTS.append(score(IT(fc.predicted_mean), f"ARIMA{best_arima_order}"))

## 9. SARIMA — weekly seasonality (s = 7)

In [ ]:
sarima_orders = [(p, 1, q) for p in range(0, 3) for q in range(0, 3)]
seasonal_orders = [(P, D, Q, SEASON) for P in range(0, 2) for D in range(0, 2) for Q in range(0, 2) if (P, D, Q) != (0, 0, 0)]
sarima_grid = grid_search(T(train), sarima_orders, seasonal_orders)
sarima_grid

In [ ]:
best_s_order, best_s_seasonal = sarima_grid.loc[0, "order"], sarima_grid.loc[0, "seasonal"]
sarima_res = SARIMAX(T(train), order=best_s_order, seasonal_order=best_s_seasonal).fit(disp=False)
print(sarima_res.summary().tables[1])
fc = sarima_res.get_forecast(h)
SARIMA_NAME = f"SARIMA{best_s_order}x{best_s_seasonal}"
FORECASTS[SARIMA_NAME] = IT(fc.predicted_mean)
SARIMA_CI = IT(fc.conf_int())
RESULTS.append(score(IT(fc.predicted_mean), SARIMA_NAME))

## 10. SARIMAX — ARIMA + calendar regressors
Regressors: holiday flag (and the day after a holiday) plus day-of-week dummies.

In [ ]:
X_train, X_test = X_all.loc[train.index], X_all.loc[test.index]
# NOTE: future holidays are known in advance (Poya calendar), so using them as exog is legitimate.

sarimax_grid = grid_search(T(train), [(p, 1, q) for p in range(0, 3) for q in range(0, 3)], exog=X_train)
best_x_order = sarimax_grid.loc[0, "order"]
sarimax_res = SARIMAX(T(train), exog=X_train, order=best_x_order).fit(disp=False)
print(sarimax_grid.head(3))
print(sarimax_res.summary().tables[1])
fc = sarimax_res.get_forecast(h, exog=X_test)
SARIMAX_NAME = f"SARIMAX{best_x_order}+calendar"
FORECASTS[SARIMAX_NAME] = IT(fc.predicted_mean)
RESULTS.append(score(IT(fc.predicted_mean), SARIMAX_NAME))

## 11. Exponential smoothing (ETS)

In [ ]:
ets_cfgs = {
    "ETS SES (A,N,N)":            dict(trend=None, seasonal=None),
    "ETS Holt damped (A,Ad,N)":   dict(trend="add", damped_trend=True, seasonal=None),
    "ETS Holt-Winters (A,Ad,A7)": dict(trend="add", damped_trend=True, seasonal="add", seasonal_periods=SEASON),
}
ETS_MODELS = {}
for name, cfg in ets_cfgs.items():
    m = ExponentialSmoothing(T(train), initialization_method="estimated", **cfg).fit(optimized=True)
    ETS_MODELS[name] = m
    FORECASTS[name] = IT(m.forecast(h))
    RESULTS.append(score(FORECASTS[name], name))
    print(f"{name:30s} AIC={m.aic:.1f}")

## 12. ML benchmark — Gradient Boosting on lag features
Recursive multi-step forecasting (each prediction feeds the next step) — the honest way to compare with ARIMA.

In [ ]:
LAGS = [1, 2, 3, 7, 14]
def build_features(series):
    X = pd.DataFrame(index=series.index)
    for l in LAGS: X[f"lag{l}"] = series.shift(l)
    X["roll7"]  = series.shift(1).rolling(7).mean()
    X["roll14"] = series.shift(1).rolling(14).mean()
    X["diff1"]  = series.shift(1) - series.shift(2)
    X["dow"]    = series.index.dayofweek
    X["holiday"] = is_holiday.reindex(series.index).fillna(0).values
    return X

# Model the CHANGE (y_t - y_{t-1}) instead of the level -> tree models can't extrapolate levels
Xtr = build_features(train)
ytr = train.diff()
mask = Xtr.notna().all(axis=1) & ytr.notna()
gbr = GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=.03, subsample=.8, random_state=RANDOM_STATE)
gbr.fit(Xtr[mask], ytr[mask])

hist = train.copy()
preds = []
for t in test.index:
    hist.loc[t] = np.nan
    x = build_features(hist).loc[[t]]
    step = gbr.predict(x)[0]
    hist.loc[t] = hist.iloc[-2] + step
    preds.append(hist.loc[t])
FORECASTS["GradientBoosting (lags)"] = pd.Series(preds, index=test.index)
RESULTS.append(score(preds, "GradientBoosting (lags)"))

imp = pd.Series(gbr.feature_importances_, index=Xtr.columns).sort_values()
imp.plot(kind="barh", figsize=(6, 3.5), title="GBR feature importance"); plt.show()

## 13. Hold-out evaluation (last 60 days, real posted days only)

In [ ]:
res_tbl = (pd.DataFrame(RESULTS).set_index("model").round(2)
           .sort_values("MAE"))
res_tbl["beats_naive"] = res_tbl["MAE"] < res_tbl.loc["Naive (last value)", "MAE"]
res_tbl

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5.5))
ctx = y.iloc[-TEST_DAYS - 90:-TEST_DAYS]
ax.plot(ctx, color="gray", label="train (last 90d)")
ax.scatter(test.index[test_obs_mask], df.loc[test.index, OBSERVED][test_obs_mask], color="black", s=18, label="actual (posted)", zorder=5)
for name in res_tbl.index[:5]:
    ax.plot(FORECASTS[name], lw=1.8, label=name)
ax.fill_between(SARIMA_CI.index, SARIMA_CI.iloc[:, 0], SARIMA_CI.iloc[:, 1], alpha=.12, color="tab:blue", label="SARIMA 95% PI")
ax.set_title("Hold-out forecasts — top 5 models by MAE"); ax.legend(fontsize=8, ncol=2); plt.show()

## 14. Rolling-origin (walk-forward) cross-validation
A single 60-day hold-out is one draw. Here every model is **refit** at many forecast origins across the last ~6 months (45 origins, every 4 days)
and scored at **1-, 7- and 14-day horizons** — this is the result to trust for model selection.

In [ ]:
HORIZONS = [1, 7, 14]
N_ORIGINS = 45
STEP = 4
origins = y.index[-(N_ORIGINS * STEP + max(HORIZONS)):-max(HORIZONS):STEP]
print(f"{len(origins)} origins from {origins[0].date()} to {origins[-1].date()}")

def fc_naive(tr, H, Xtr=None, Xf=None):  return np.repeat(tr.iloc[-1], H)
def fc_snaive(tr, H, Xtr=None, Xf=None): return np.resize(tr.iloc[-SEASON:].values, H)
def fc_arima(tr, H, Xtr=None, Xf=None):  return IT(SARIMAX(T(tr), order=best_arima_order).fit(disp=False).forecast(H).values)
def fc_sarima(tr, H, Xtr=None, Xf=None): return IT(SARIMAX(T(tr), order=best_s_order, seasonal_order=best_s_seasonal).fit(disp=False).forecast(H).values)
def fc_sarimax(tr, H, Xtr=None, Xf=None):return IT(SARIMAX(T(tr), exog=Xtr, order=best_x_order).fit(disp=False).forecast(H, exog=Xf).values)
def fc_holt(tr, H, Xtr=None, Xf=None):   return IT(ExponentialSmoothing(T(tr), trend="add", damped_trend=True, initialization_method="estimated").fit().forecast(H).values)

CV_MODELS = {
    "Naive": fc_naive, "Seasonal naive": fc_snaive,
    f"ARIMA{best_arima_order}": fc_arima, SARIMA_NAME: fc_sarima,
    SARIMAX_NAME: fc_sarimax, "ETS Holt damped": fc_holt,
}

rows = []
t0 = time.time()
for o in origins:
    tr = y.loc[:o]
    H = max(HORIZONS)
    fut = pd.date_range(o + pd.Timedelta(days=1), periods=H, freq="D")
    for name, f in CV_MODELS.items():
        try:
            p = f(tr, H, X_all.loc[tr.index], X_all.loc[fut])
        except Exception as e:
            continue
        for k in range(H):
            d = fut[k]
            if is_obs.get(d, False):
                rows.append({"model": name, "origin": o, "h": k + 1, "actual": df.loc[d, OBSERVED], "pred": p[k]})
print(f"CV done in {time.time() - t0:.0f}s")
cv = pd.DataFrame(rows)
cv["abs_err"] = (cv["actual"] - cv["pred"]).abs()
cv["sq_err"] = (cv["actual"] - cv["pred"]) ** 2

In [ ]:
def cv_table(hmax):
    sub = cv[cv["h"] <= hmax]
    t = sub.groupby("model").agg(MAE=("abs_err", "mean"), RMSE=("sq_err", lambda s: np.sqrt(s.mean())), n=("abs_err", "size"))
    t["MAPE%"] = sub.assign(ape=sub["abs_err"] / sub["actual"] * 100).groupby("model")["ape"].mean()
    t["skill_vs_naive%"] = (1 - t["MAE"] / t.loc["Naive", "MAE"]) * 100
    return t.round(2).sort_values("MAE")

for hmax in HORIZONS:
    print(f"\n=== Walk-forward CV, horizons 1..{hmax} days ===")
    display(cv_table(hmax))

In [ ]:
curve = cv.groupby(["model", "h"])["abs_err"].mean().unstack(0)
curve.plot(marker="o", figsize=(12, 4.5), title="Walk-forward MAE by forecast horizon")
plt.xlabel("days ahead"); plt.ylabel("MAE (Rs/kg)"); plt.show()

## 15. Residual diagnostics — best statistical model

In [ ]:
cv14 = cv_table(14)
stat_models = [m for m in cv14.index if m not in ("Naive", "Seasonal naive")]
BEST = stat_models[0]
print("Best statistical model on 14-day walk-forward CV:", BEST)

def fit_full(name, series, X=None):
    if name.startswith("SARIMAX"): return SARIMAX(series, exog=X, order=best_x_order).fit(disp=False)
    if name.startswith("SARIMA"):  return SARIMAX(series, order=best_s_order, seasonal_order=best_s_seasonal).fit(disp=False)
    if name.startswith("ARIMA"):   return SARIMAX(series, order=best_arima_order).fit(disp=False)
    return ExponentialSmoothing(series, trend="add", damped_trend=True, initialization_method="estimated").fit()

best_train_res = fit_full(BEST, T(train), X_train)   # residuals on model scale
resid = pd.Series(best_train_res.resid, index=train.index).iloc[10:]
if hasattr(best_train_res, "plot_diagnostics"):
    best_train_res.plot_diagnostics(figsize=(14, 8)); plt.show()
lb = acorr_ljungbox(resid, lags=[7, 14, 21], return_df=True)
print("Ljung-Box (p > 0.05 => residuals ~ white noise):"); print(lb.round(4))
print("Residual mean: {:.2f}   std: {:.2f}".format(resid.mean(), resid.std()))

## 16. Final model — refit on all data and forecast the next 30 days

In [ ]:
future_idx = pd.date_range(y.index[-1] + pd.Timedelta(days=1), periods=FORECAST_DAYS, freq="D")
X_future = make_exog(future_idx)   # holidays unknown here -> 0; set known Poya days to 1 for better forecasts

final = fit_full(BEST, T(y), X_all)
if hasattr(final, "get_forecast"):
    f = final.get_forecast(FORECAST_DAYS, exog=X_future if BEST.startswith("SARIMAX") else None)
    fc_mean, ci = IT(f.predicted_mean), IT(f.conf_int(alpha=.05))
    ci80 = IT(f.conf_int(alpha=.20))
else:
    fc_mean = IT(final.forecast(FORECAST_DAYS))
    sims = IT(final.simulate(FORECAST_DAYS, repetitions=1000, error="add"))
    ci = pd.DataFrame({"lower": sims.quantile(.025, axis=1), "upper": sims.quantile(.975, axis=1)}, index=future_idx)
    ci80 = pd.DataFrame({"lower": sims.quantile(.10, axis=1), "upper": sims.quantile(.90, axis=1)}, index=future_idx)

forecast_df = pd.DataFrame({"forecast": fc_mean.values,
                            "lo80": ci80.iloc[:, 0].values, "hi80": ci80.iloc[:, 1].values,
                            "lo95": ci.iloc[:, 0].values, "hi95": ci.iloc[:, 1].values}, index=future_idx).round(1)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y.iloc[-120:], color="gray", label="history (interpolated)")
ax.scatter(obs.iloc[-80:].index, obs.iloc[-80:], s=10, color="black", label="posted")
ax.plot(forecast_df["forecast"], color="tab:red", lw=2, label=f"forecast — {BEST}")
ax.fill_between(future_idx, forecast_df.lo95, forecast_df.hi95, color="tab:red", alpha=.12, label="95% PI")
ax.fill_between(future_idx, forecast_df.lo80, forecast_df.hi80, color="tab:red", alpha=.22, label="80% PI")
ax.set_title(f"Next {FORECAST_DAYS} days live broiler price forecast"); ax.legend(); plt.show()
forecast_df

## 17. Summary to share

In [ ]:
print("DATA")
print(f"  {len(y)} days, {is_obs.sum()} posted ({is_obs.mean():.0%}), {int(is_holiday.sum())} holidays")
print(f"  price range {obs.min():.0f}-{obs.max():.0f} Rs/kg, mean {obs.mean():.0f}")
print("\nPREPROCESSING"); print(prep_summary.to_string()); print(imp_tbl.to_string())
print("\nSTATIONARITY"); print(st.to_string(index=False))
print("\nSELECTED ORDERS")
print(f"  ARIMA  : {best_arima_order}   (AIC {arima_grid.loc[0,'AIC']:.1f})")
print(f"  SARIMA : {best_s_order} x {best_s_seasonal}   (AIC {sarima_grid.loc[0,'AIC']:.1f})")
print(f"  SARIMAX: {best_x_order} + calendar exog   (AIC {sarimax_grid.loc[0,'AIC']:.1f})")
print("\nHOLD-OUT (last 60 days)"); print(res_tbl.to_string())
for hmax in HORIZONS:
    print(f"\nWALK-FORWARD CV h<= {hmax}"); print(cv_table(hmax).to_string())
print("\nBEST MODEL:", BEST)
print("Ljung-Box:"); print(lb.round(4).to_string())
print("\nFORECAST (first 14 days)"); print(forecast_df.head(14).to_string())

# save artefacts
res_tbl.to_csv("holdout_results.csv")
cv.to_csv("walkforward_cv_raw.csv", index=False)
forecast_df.to_csv("forecast_next_30_days.csv")
print("\nSaved: holdout_results.csv, walkforward_cv_raw.csv, forecast_next_30_days.csv")

### How to read the results
* **MASE < 1** means the model beats a one-step naive forecast on average; **skill_vs_naive% > 0** in the CV table means it beats
  "tomorrow = today" over the same horizon.
* Broiler prices behave close to a **random walk with sudden supply-driven jumps**, so beating naive at 1 day is hard; the gain from
  ARIMA/SARIMA usually appears at the 7–14 day horizon.
* Prefer the model that wins the **walk-forward CV** (section 14) over a single hold-out, and check that its Ljung-Box p-values are > 0.05.
* Prediction intervals matter more than the point forecast for planning — use the 80% band for operational decisions.